# AndroidMalSet2026: loading the converted files

This notebook runs in Jupyter Notebook (Anaconda) or Google Colab. It expects the output folder of `chuyen_dinh_dang.py` with the `npz` format (`X.npz`, `y.npy`, `classes.txt`, `feature_names.txt`, `sample_ids.csv`). On Google Colab, upload the folder to Google Drive and mount it first.

In [ ]:
# Google Colab only: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = 'xuat'   # folder written by chuyen_dinh_dang.py

In [ ]:
import numpy as np, pandas as pd
from scipy import sparse

X = sparse.load_npz(f'{DATA_DIR}/X.npz')          # CSR matrix, float32
y = np.load(f'{DATA_DIR}/y.npy')                    # class index per row
classes = open(f'{DATA_DIR}/classes.txt', encoding='utf-8').read().splitlines()
features = open(f'{DATA_DIR}/feature_names.txt', encoding='utf-8').read().splitlines()
ids = pd.read_csv(f'{DATA_DIR}/sample_ids.csv')
print(X.shape, len(features), classes)
print(pd.Series(y).map(dict(enumerate(classes))).value_counts())

## scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42).fit(X_tr, y_tr)
print(classification_report(y_te, rf.predict(X_te), target_names=classes, digits=4))

## Keras / TensorFlow

The matrix stays sparse in memory; each mini-batch is converted to a dense array only when it is fed to the network.

In [ ]:
import tensorflow as tf

class SparseBatches(tf.keras.utils.PyDataset):
    def __init__(self, X, y, batch_size=256, **kw):
        super().__init__(**kw)
        self.X, self.y, self.bs = X, y, batch_size
    def __len__(self):
        return int(np.ceil(self.X.shape[0] / self.bs))
    def __getitem__(self, i):
        s = slice(i * self.bs, (i + 1) * self.bs)
        return np.log1p(self.X[s].toarray()), self.y[s]

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(classes), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(SparseBatches(X_tr, y_tr), validation_data=SparseBatches(X_te, y_te), epochs=5)

## PyTorch

A batch can be turned into a tensor with `torch.from_numpy(X[s].toarray())`, or kept sparse with `torch.sparse_csr_tensor`.

## Weka and LibSVM

`AndroidMalSet2026.arff` opens in the Weka Explorer (Open file). With 20,032 attributes, start Weka with more memory, for example `java -Xmx8g -jar weka.jar`. `AndroidMalSet2026.libsvm` can be read with `sklearn.datasets.load_svmlight_file`, XGBoost (`xgboost.DMatrix('AndroidMalSet2026.libsvm?format=libsvm')`), LightGBM or the LIBSVM tools.